# Machine (Un)learning

<img src="https://live.staticflickr.com/65535/54548177187_3b31449095_b.jpg" style="height: 450px;"/>

*Image generated using the DALL-E model.*

## Introduction

### Motivation

Machine unlearning is a fascinating and increasingly popular topic related to learning algorithms, primarily deep learning networks. With the growing number of parameters and model sophistication, their ability to remove outdated, incorrect, or sensitive information (e.g., related to user privacy) becomes crucial. Unlearning allows us to *erase from the model's memory* selected information in a way that minimizes the associated damage to other learned data. This allows us to achieve more precise and safer models, without the constant need to retrain models from scratch.

### Benefits

Introducing unlearning into the world of deep learning can bring many benefits. It allows, among other things, to eliminate the model's ability to generate dangerous content. For example, if a generative model was trained on a huge set of images, but some of them contain inappropriate content, such as nudity, unlearning can be applied to remove the concept of nudity from the model's memory, which will reduce the likelihood of generating similar images in the future.

Secondly, in connection with data protection regulations (such as GDPR, i.e., General Data Protection Regulation in the European Union), users have the right to withdraw consent for their data to be processed by a given company. However, if the model was trained using this data, it is not trivial to get rid of this knowledge without needing to retrain the model.

Finally, unlearning can help eliminate biases and errors that may appear in models, leading to fairer and more accurate results. Machine learning models can sometimes absorb biases and inequalities contained in the training data (e.g., related to gender or skin color), resulting in biased predictions. Unlearning allows us to remove these biases, improving the quality and fairness of results.

*<u>Note</u>: In the classic machine unlearning problem, we don't necessarily want the model to be unable to predict a given class, but rather to behave as if that data was not in the training set (meaning resistance to membership inference attacks). Hence, the desired property is often not directly minimizing effectiveness on a certain portion of data, but rather striving for the model to behave towards them randomly or as it would towards data that was not actually in the training set. For the purposes of our task, we simplify this goal and want the model to simply predict a given class with the lowest possible accuracy.*

---

## Problem Description

For our adventure with unlearning, we will analyze a classification problem on a standard computer vision dataset, Fashion MNIST. We will use the classic convolutional architecture LeNet to solve this problem (visualization and description below). We are provided with a base model that has been trained on the entire dataset, i.e., on all ten classes of the Fashion MNIST dataset (t-shirt, trousers, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot). Below is the code to load this model.

### Your Task

Your task is to unlearn a selected class of the Fashion MNIST dataset from the base model. We require that the last layer (classification layer) remains **intact**, meaning your intervention should be at the feature extractor level, not through mechanical modification of the signal responsible for generating logits for individual classes. The intervention should focus solely on the level of earlier layers of the model, excluding any direct change in the model's classifying function itself.

### Evaluation

To check how well you have done with this task, we have prepared a set of metrics that will allow us to assess the quality of your solution.

You will be evaluated on four aspects:
- **Unlearning classification of the selected class** - after all, that's what unlearning is about! However, you do not know which class's unlearning you will be finally evaluated on within the test set. For validation purposes, we have adopted one of the ten classes of the Fashion MNIST dataset in this notebook, but remember that you will ultimately be evaluated on a different class!
- **Maintaining high performance on the remaining classes** - the model must stop working only on the selected class, but must not lose performance on the remaining classes - here accuracy must be as high as possible!
- **Intervention in the base model** - try to keep the modification of the base model's weights as small as possible!
- **Diversity of predictions for the forgotten class** - the idea is for the model to be resistant to the previously mentioned membership inference attack, i.e., so that it is impossible to tell that the model ever saw the data it is supposed to unlearn!

---

### Detailed Description:

#### Unlearning the i-th class, $i \in \lbrace 1, ..., 10 \rbrace$.

The process of unlearning the i-th class (one of ten possible classes) consists of **removing the model's ability to recognize and classify data belonging to this particular class**. This is a key aspect of unlearning, which allows for the elimination of unwanted or inappropriate data from the model.
Specifically, we can measure **accuracy** on data **belonging to a specific** class number i as $\Psi_{i}$:

$$ \Psi_{i} = \frac{TP_i}{TP_i + FN_i}, $$

where $TP_i$ denotes the number of examples correctly classified as class $i$ (true positive rate), and $FN_i$ denotes the number of examples from class $i$ that were incorrectly classified as belonging to another class (false negative rate).
In our problem, we want to **minimize** the value of $\Psi_{i}$.

#### Maintaining high performance on the remaining classes

An important aspect of unlearning is ensuring that **removing the i-th class does not negatively affect the model's performance with respect to the remaining classes**. The model must still be able to achieve high classification accuracy on all remaining classes. This is a challenge because the unlearning process can disturb the model's balance and negatively affect its feature extraction ability, also with respect to the remaining classes. The set of data that should still be well classified by the model is called simply the remain dataset. We can evaluate the model's classification accuracy on this data by measuring it for all classes except class number i and define this as a function $\Phi$ depending on $i$.

Specifically, we can use the accuracy on the remain set as $\Phi_{i}$:
$$
\Phi_{i} = \frac{\sum_{k \neq i} TP_k}{\sum_{k \neq i} \left( TP_k + FN_k \right)},
$$
where: $TP_k$ is the number of correctly classified examples belonging to class $k$, excluding class $i$ (true positive rate), $FN_k$ is the number of examples incorrectly not classified as class $k$, excluding class $i$ (false negative rate). In other words, the numerator contains the number of correctly classified examples from all classes except class number $i$, and the denominator contains the number of examples belonging to all classes except the $i$-th class.

#### Intervention in the base model

The unlearning process involves intervening in the base model, which can have various consequences. It is important that this **intervention is minimal** and does not lead to model destabilization. In practice, this means that changes introduced to the model should be limited to necessary modifications that allow removing the i-th class without affecting the structure and functionality of the model, thus with minimal modification of the model's parameters. We require that the **last layer in the LeNet model remains intact**, but we will also measure the distance between the output and the unlearned model proposed by you. This is also a way to better understand the model's operation. If the distance from the base model is not too large, and we trust the base model, then the model you obtain is close enough to inspire our confidence.

In our problem, we will use the traditional **$\mathbf{L_2}$ distance**, which can be expressed by the formula:

$$ \mathcal{d}_{dist} = \mathcal{L_2} (\theta; \theta_0) = \sqrt{\sum_j (\theta_j - \theta_{0,j})^2}, $$

where
- $\theta$ denotes the parameter vector of the model after the unlearning process,
- $\theta_0$ denotes the parameter vector of the base model,
- $\theta_j$ and $\theta_{0,j}$ are the respective parameter values for the j-th layer of the models.

The $\mathcal{L}_2$ distance sums the squares of the distance differences between successive layers of the base model with respect to the corresponding layers of the modified model, and finally takes the square root of this sum. The smaller the value of $\mathcal{d}_{dist}$, the smaller the intervention in the base model, which is desirable in the context of minimizing changes in the model structure.

#### Diversity of predictions for the forgotten class

The last aspect of the evaluation is the **diversity of predictions for the forgotten class**. After the unlearning process, the model should generate diverse predictions for data belonging to class $i$ that the model was supposed to forget. This means that the model should simply assign them to other classes in a varied way so that it is not easy to tell whether this data was originally in the base model's training set.

The model should also not assign all predictions to one specific class, as this could be interpreted as merging selected classes, which is not desired.

To measure prediction diversity, we will use **Kullback-Leibler divergence (KL)**. This is a standard measure of divergence between two probability distributions. In our case, we examine the distance of the model's prediction distribution from the uniform distribution, which means that none of the remaining classes should be favored.

KL divergence is defined as:

$$
\mathcal{D}_{KL}(p \parallel u) = \sum_{c=1}^{C} p(c) \log \left( \frac{p(c)}{u(c)} \right),
$$

where:
- $p(c)$ is the probability assigned by the model to class $c$ for a sample,
- $u(c)$ is the probability according to the uniform distribution, i.e., $u(c) = \frac{1}{C}$,
- $C$ is the number of all classes.

In the case of ideal prediction diversity for the forgotten class, the distribution $p(c)$ should be as close as possible to the uniform distribution $u(c)$, which means a minimal value of $\mathcal{D}_{KL}$.

If KL divergence is still quite complicated, don't worry - in the **Supplementary Information** section, we have provided some helpful tips. Also, take a look at the code defining the function - we left some comments there.

---

We will evaluate all of the above four objectives on the Fashion MNIST data, **but you do not know which class we will ultimately want to unlearn**. **Furthermore, you do not know which data subset from individual classes will be used for unlearning.** To prepare your solution, you can choose any class, but the solution must be ready to unlearn each class of this dataset. While the data distribution in the test set will be similar to the distribution in this notebook, try to avoid overfitting to the selected data.

So check if your solution is universal. Make sure it works for different labels designated for forgetting and different datasets (the architecture is fixed, adapted to processing visual data of dimension $28 \times 28$).

#### Final score

Overall, your result can be mathematically written as a weighted sum:
$$
\Sigma_{score} = \frac{1}{4} \cdot \Sigma_{target} + \frac{1}{4} \cdot \Sigma_{remain} + \frac{1}{4} \cdot \Sigma_{dist} + \frac{1}{4} \cdot \Sigma_{kl}
$$

*<u>Note</u>: We would like to maximize the metric $\Phi$, and minimize the others, i.e., $\Psi$, $d_{dist}$, and $\mathcal{D}_{KL}$. Hence, pay attention to the order of subtraction when calculating the individual $\Sigma$ components for the final evaluation!*

$\Sigma_{target}$ – score for unlearning effectiveness on the selected class (for simplicity, in the task we assume that the lower the effectiveness on the target class, the better; scaled in the range [0, 100] according to thresholds $[0.09, 0.3]$):

$$
\Sigma_{target}(\Psi_{i}) = 
\begin{cases}
100 & \text{if } \Psi_{i} \leq 0.09 \\
0 & \text{if } \Psi_{i} \geq 0.3 \\
100 \cdot \dfrac{0.3 - \Psi_{i}}{0.3 - 0.09} & \text{otherwise}
\end{cases}
$$

$\Sigma_{other}$ – score for maintaining accuracy on the remaining classes (the higher the effectiveness, the higher the score for this metric; scaled in the range [0, 100] according to thresholds $[0.87, 0.90]$):

$$
\Sigma_{other}(\Phi_{i}) = 
\begin{cases}
100 & \text{if } \Phi_{i} \geq 0.90 \\
0 & \text{if } \Phi_{i} \leq 0.87 \\
100 \cdot \dfrac{\Phi_{i} - 0.87}{0.90 - 0.87} & \text{otherwise}
\end{cases}
$$

$\Sigma_{dist}$ – score for the degree of intervention in the model (the smaller the $L_2$ distance, the higher the score for this metric; scaled in the range [0, 100] according to thresholds $[1.3, 3.0]$):

$$
\Sigma_{dist}(\mathcal{d}_{dist}) = 
\begin{cases}
100 & \text{if } \mathcal{d}_{dist} \leq 1.3 \\
0 & \text{if } \mathcal{d}_{dist} \geq 3.0 \\
100 \cdot \dfrac{3.0 - \mathcal{d}_{dist}}{3.0 - 1.3} & \text{otherwise}
\end{cases}
$$

$\Sigma_{kl}$ – score for prediction diversity (the lower the KL divergence, the higher the score for this metric; scaled in the range [0, 100] according to thresholds $[0.2, 0.5]$):

$$
\Sigma_{kl}(\mathcal{D}_{KL}) = 
\begin{cases}
100 & \text{if } \mathcal{D}_{KL} \leq 0.2 \\
0 & \text{if } \mathcal{D}_{KL} \geq 0.5 \\
100 \cdot \dfrac{0.5 - \mathcal{D}_{KL}}{0.5 - 0.2} & \text{otherwise}
\end{cases}
$$

**However**, the task is graded **0 points in all categories** if your solution:
- does not follow the guidelines, for example:
    - introduces a change in the network architecture;
    - modifies the last layer of the network;
    - makes any attempt to cheat, e.g., by modifying the evaluation function;
- the solution is unsatisfactory:
    - the classification accuracy on the set of the class to forget remains above 0.5;
    - the classification accuracy on the set of remaining classes drops below 0.75;
    - the $L_2$ distance is greater than 8.0;
    - the Kullback-Leibler divergence value exceeds 1.75.

Furthermore, the unlearning process you propose, i.e., the operation of the *unlearn()* function, may take no longer than 5 minutes using a GPU.

You can receive a maximum of 100 points for the task.

Remember that during verification, the flag *FINAL_EVALUATION_MODE* will be set to True.

Good luck!

---

## Starter Code

In [1]:
######################### DO NOT CHANGE THIS CELL ##########################

# If necessary, import additional libraries below, in your own code.

import os
from copy import deepcopy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import tarfile
import tempfile

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
######################### DO NOT CHANGE THIS CELL ##########################

FINAL_EVALUATION_MODE = False  # During verification, we will set this flag to True.

In [3]:
######################### DO NOT CHANGE THIS CELL ##########################

seed = 101

os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(seed)

### Data

In [4]:
######################### DO NOT CHANGE THIS CELL ##########################

class FilteredFashionMNIST(datasets.FashionMNIST):
    def __init__(self, *args, classes=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.classes = classes
        if self.classes is not None:
            self.data, self.targets = self._filter_classes(
                self.data, self.targets, self.classes)

    def _filter_classes(self, data, targets, classes):
        mask = torch.zeros_like(targets, dtype=torch.bool)
        for c in classes:
            mask = mask | (targets == c)
        return data[mask], targets[mask]

In [5]:
######################### DO NOT CHANGE THIS CELL ##########################

BATCH_SIZE = 64

tempdir = tempfile.TemporaryDirectory()
TMP_DIR = tempdir.name
DATA_PATH = os.path.join(TMP_DIR, "data")

if not FINAL_EVALUATION_MODE:
    import gdown

    GDRIVE_DATA = [
        ("1SeXrzvs64MBG57ayK6965cya3WRfCMx_", "data/FashionMNIST.tar.gz"),
        ("1mqzxg-_0PJjvErvfwOGad6siNcQKqwk5", "data/FilteredFashionMNIST.tar.gz"),
        ("1YSn8EFjbYDcDCVdlByA_kDKCg4VZLwH8", "models/lenet_base_final.pt"),
    ]
    
    for file_id, output in GDRIVE_DATA:        
        url = f'https://drive.google.com/uc?id={file_id}'
        os.makedirs(os.path.dirname(output), exist_ok=True)
        gdown.download(url, output, quiet=False)
        print(f"Downloaded: {output}")

Downloading...
From (original): https://drive.google.com/uc?id=1SeXrzvs64MBG57ayK6965cya3WRfCMx_
From (redirected): https://drive.google.com/uc?id=1SeXrzvs64MBG57ayK6965cya3WRfCMx_&confirm=t&uuid=e4babf9a-83e7-40a5-9671-a9ba0ecbc6cd
To: c:\Users\raian\source\repos\AI\IOAI_prep\poland\2_3\unlearning\data\FashionMNIST.tar.gz
100%|██████████| 61.8M/61.8M [00:06<00:00, 9.77MB/s]


Downloaded: data/FashionMNIST.tar.gz


Downloading...
From (original): https://drive.google.com/uc?id=1mqzxg-_0PJjvErvfwOGad6siNcQKqwk5
From (redirected): https://drive.google.com/uc?id=1mqzxg-_0PJjvErvfwOGad6siNcQKqwk5&confirm=t&uuid=6b0a4c2c-380a-46b2-88a1-6b236229a35f
To: c:\Users\raian\source\repos\AI\IOAI_prep\poland\2_3\unlearning\data\FilteredFashionMNIST.tar.gz
100%|██████████| 61.8M/61.8M [00:06<00:00, 9.50MB/s]


Downloaded: data/FilteredFashionMNIST.tar.gz


Downloading...
From: https://drive.google.com/uc?id=1YSn8EFjbYDcDCVdlByA_kDKCg4VZLwH8
To: c:\Users\raian\source\repos\AI\IOAI_prep\poland\2_3\unlearning\models\lenet_base_final.pt
100%|██████████| 185k/185k [00:00<00:00, 1.65MB/s]


Downloaded: models/lenet_base_final.pt


In [6]:
######################### DO NOT CHANGE THIS CELL ##########################

def unpack_tar_gz(filename: str, path: str = DATA_PATH) -> None:
    """Unpacks a tar.gz archive to the specified directory."""
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall(path=path)

unpack_tar_gz("data/FashionMNIST.tar.gz", os.path.join(DATA_PATH, "FashionMNIST"))
unpack_tar_gz("data/FilteredFashionMNIST.tar.gz", os.path.join(DATA_PATH, "FilteredFashionMNIST"))

transform_fashion = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])

def get_data_dict():
    def create_filtered_loader(classes):
        dataset = FilteredFashionMNIST(
            root=DATA_PATH,
            download=True,
            transform=transform_fashion,
            classes=classes)
        loader = DataLoader(
            dataset=dataset,
            batch_size=BATCH_SIZE,
            shuffle=True)
        return loader

    loader_fashion = DataLoader(
        dataset=datasets.FashionMNIST(
            root=DATA_PATH,
            download=True,
            transform=transform_fashion),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    class_groups = {}
    num_classes = 10
    for i in range(num_classes):
        class_groups[f"{i}"] = [i]
        class_groups[f"~{i}"] = [j for j in range(num_classes) if j != i]

    data_dict = {
        "fashion": {
            "loader": loader_fashion,
        }
    }

    for group_name, classes in class_groups.items():
        loader = create_filtered_loader(classes)
        data_dict["fashion"][f"loader_{group_name}"] = loader
    return data_dict

data_dict = get_data_dict()

### Model

In [7]:
######################### DO NOT CHANGE THIS CELL ##########################

class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(6),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        latent_dim = 256
        self.fc = nn.Linear(latent_dim, 120)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        out = self.block1(x)
        out = self.block2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc(out)
        out = self.relu(out)
        out = self.fc1(out)
        out = self.relu1(out)
        out = self.fc2(out)
        return out

<img src="https://live.staticflickr.com/65535/54549285793_e078de98d3_b.jpg" style="height: 450px;"/>

Source: GeeksForGeeks

### Loading the model

In [80]:
pretrained_model = LeNet(num_classes=10)
initial_state_dict = torch.load("./models/lenet_base_final.pt")
pretrained_model.load_state_dict(initial_state_dict)
pretrained_model = pretrained_model.to(DEVICE)

### Metrics

In [9]:
######################### DO NOT CHANGE THIS CELL ##########################

def evaluate_target(model, data_loader, criterion, device="cpu"):
    """
    Function evaluating model performance on target data.
    
    :param model: PyTorch model to evaluate.
    :param data_loader: DataLoader containing data for classification accuracy evaluation.
    :param criterion: Loss function used to calculate the loss.
    :param device: Device on which calculations are performed (default "cpu").
    :return: Tuple containing average loss and accuracy on target data.
    """
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return total_loss / len(data_loader.dataset), correct / len(data_loader.dataset)

def measure_target_uniformity(model, loader_target, device="cpu", num_classes=10):
    """
    Measures how close the model's predictions for the target class are to a uniform distribution.
    We use KL divergence: KL(U || p(x)) or KL(p(x) || U) as a measure of uniformity.

    We calculate the average KL(p || U) = sum_{y} p(y) log [p(y) / (1/num_classes)].
    Lower KL value means greater uniformity.
    Detailed explanation below in supplementary information.
    """
    model.eval()
    kl_sum = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, _ in loader_target:
            images = images.to(device)
            logits = model(images)
            log_probs = F.log_softmax(logits, dim=1)
            probs = torch.exp(log_probs)
            entropy_term = (probs * log_probs).sum(dim=1)
            kl_batch = entropy_term + torch.log(
                torch.tensor(num_classes, device=device)
            )
            kl_sum += kl_batch.sum().item()
            total_samples += images.size(0)

    return kl_sum / total_samples
    
def measure_l2_distance(model, initial_state_dict, device="cpu"):
    """
    Measures the L_2 distance between the current state of the model and its initial state.
    """
    l2_distance = 0.0
    for name, param in model.named_parameters():
        if param.requires_grad:
            initial_param = initial_state_dict[name]
            l2_distance += torch.sum((param.to(device) - initial_param.to(device)) ** 2).item()
    return l2_distance ** 0.5

In [10]:
######################### DO NOT CHANGE THIS CELL ##########################

WEIGHTS = [0.25, 0.25, 0.25, 0.25]

acc_forget_thresholds = [0.09, 0.3]
acc_other_thresholds = [0.87, 0.90]
dist_thresholds = [1.3, 3.0]
d_kl_thresholds = [0.2, 0.5]

acc_forget_absolute_threshold = 0.5
acc_target_absolute_threshold = 0.75
distance_absolute_threshold = 8.0
dkl_absolute_threshold = 1.75

def compute_final_score(
        kl_after, acc_rest_after, l2_dist, acc_target_after
    ):
    """
    Function calculates the final score based on various metrics.
    :param kl_after: KL value after training.
    :param acc_rest_after: Accuracy on the data that the model should remember.
    :param l2_dist: L2 distance after training.
    :param acc_target_after: Accuracy on the data that the model should forget.
    :return: Final score."""

    def compute_metric_score(value, thresholds, maximize=True):
        if maximize:
            if value >= thresholds[1]:
                return 100
            elif value <= thresholds[0]:
                return 0
            else:
                return 100 * (value - thresholds[0]) / (thresholds[1] - thresholds[0])
        else:
            if value <= thresholds[0]:
                return 100
            elif value >= thresholds[1]:
                return 0
            else:
                return 100 * (thresholds[1] - value) / (thresholds[1] - thresholds[0])
    print(f"KL divergence: {kl_after:.2f}")
    print(f"Classification accuracy on the remain set: {acc_rest_after:.2f}")
    print(f"L2 distance: {l2_dist:.2f}")
    print(f"Accuracy on the forget set: {acc_target_after:.2f}")

    if acc_rest_after < acc_target_absolute_threshold or acc_target_after > acc_forget_absolute_threshold \
            or l2_dist > distance_absolute_threshold or kl_after > dkl_absolute_threshold:
        return 0

    kl_score = compute_metric_score(kl_after, d_kl_thresholds, maximize=False)
    acc_rest_score = compute_metric_score(acc_rest_after, acc_other_thresholds, maximize=True)
    l2_dist_score = compute_metric_score(l2_dist, dist_thresholds, maximize=False)
    acc_target_score = compute_metric_score(acc_target_after, acc_forget_thresholds, maximize=False)

    total_score = (
        WEIGHTS[0] * kl_score +
        WEIGHTS[1] * acc_rest_score +
        WEIGHTS[2] * l2_dist_score +
        WEIGHTS[3] * acc_target_score
    )

    return total_score
    
def evaluate_model(model, data_loader_target, data_loader_rest, initial_state_dict, device):
    """
    Function evaluates the model on target and remain data, checks L2 distance
    and similarity to uniform distribution.
    :param model: model to evaluate.
    :param data_loader_target: DataLoader for target data.
    :param data_loader_rest: DataLoader for remain data.
    :param initial_state_dict: initial state of the model.
    :param criterion: loss function.
    :param device: device for calculations.
    :return: model score.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    _, acc_rest = evaluate_target(model, data_loader_rest, criterion, device)
    _, acc_target = evaluate_target(model, data_loader_target, criterion, device)
    l2_dist = measure_l2_distance(model, initial_state_dict, device)
    kl = measure_target_uniformity(model, data_loader_target, device, num_classes=10)
    return compute_final_score(kl, acc_rest, l2_dist, acc_target)

### Compliance

In [11]:
######################### DO NOT CHANGE THIS CELL ##########################

def has_same_last_layer(model1: LeNet, model2: LeNet) -> bool:
    return all(torch.equal(p1, p2) for p1, p2 in zip(model1.fc2.parameters(), model2.fc2.parameters()))

def has_same_architecture(model: LeNet) -> bool:
    return set([k for (k, v) in list(model.named_parameters())]) == \
        {'block1.0.weight', 'block1.0.bias', 'block1.1.weight', 'block1.1.bias', \
         'block2.0.weight', 'block2.0.bias', 'block2.1.weight', 'block2.1.bias', \
         'fc.weight', 'fc.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'}

### Dummy Solution

In [12]:
def dumb_solution(model, data, target_class):
    """
    Function that adds noise to model weights to change its behavior on target data.
    :param model: PyTorch model to modify.
    :param data: Data on which to modify the model.
    :param target_class: Target class on which to modify the model.
    """
    new_model = deepcopy(model)  # Create a copy of the model so as not to modify the original
    with torch.no_grad():
        for name, param in new_model.named_parameters():
            if "fc2" not in name:  # The last layer remains unchanged
                param.add_(torch.randn_like(param) * 0.1)  # Add noise to weights
    return new_model

#### Dummy solution - evaluation

In [13]:
if not FINAL_EVALUATION_MODE:
    perturbed_model = dumb_solution(pretrained_model, None, None)
    assert has_same_last_layer(pretrained_model, perturbed_model), "The last layer of the model is not the same as in the original model."
    assert has_same_architecture(perturbed_model), "The architecture of the model is not the same as in the original model."

    target_class = 9
    print(f"Target class: {target_class}")
    target_loader = data_dict["fashion"][f"loader_{target_class}"]
    other_loader = data_dict["fashion"][f"loader_~{target_class}"]

    score = evaluate_model(perturbed_model, target_loader, other_loader, initial_state_dict, DEVICE)
    print(f"Score: {score:.2f}")

Target class: 9
KL divergence: 2.06
Classification accuracy on the remain set: 0.50
L2 distance: 20.73
Accuracy on the forget set: 0.82
Score: 0.00


## Supplementary Information

#### Explanation of Kullback-Leibler Divergence

Kullback-Leibler divergence (KL) is a measure of the divergence between two probability distributions. In the context of machine learning and statistics, KL divergence allows us to assess how much one distribution differs from another. Formally, for two discrete distributions $P$ and $Q$, it is defined as:

$$
D_{KL}(P \parallel Q) = \sum_{x \in X} P(x) \log \frac{P(x)}{Q(x)}
$$

Where:
- $P(x)$ is the true distribution (e.g., real data),
- $Q(x)$ is the approximated distribution (e.g., model prediction),
- $X$ is the event space.

KL divergence is not a symmetric measure, meaning $D_{KL}(P \parallel Q) \neq D_{KL}(Q \parallel P)$. The divergence value is 0 when both distributions are identical.

##### Examples of discrete distribution comparisons:

1. **Example 1: Identical distributions**
    - $P = [0.4, 0.6]$
    - $Q = [0.4, 0.6]$
    - $D_{KL}(P \parallel Q) = 0$

2. **Example 2: Slightly different distributions**
    - $P = [0.4, 0.6]$
    - $Q = [0.5, 0.5]$
    - $D_{KL}(P \parallel Q) \approx 0.02$

3. **Example 3: Very different distributions**
    - $P = [0.9, 0.1]$
    - $Q = [0.1, 0.9]$
    - $D_{KL}(P \parallel Q) \approx 0.75$.

##### Visualization of KL Divergence

Below is Python code that visualizes KL divergence for two discrete distributions.

```python
import numpy as np
import matplotlib.pyplot as plt

def kl_divergence(p, q):
     """Calculates KL divergence between two distributions."""
     p = np.array(p)
     q = np.array(q)
     return np.sum(p * np.log(p / q))

# Example distributions
P = [0.4, 0.6]
Q_list = [
     [0.4, 0.6],  # Identical distribution
     [0.5, 0.5],  # Slightly different
     [0.9, 0.1]   # Very different
]

# Calculate KL divergence
kl_values = [kl_divergence(P, Q) for Q in Q_list]

# Visualization
labels = ['Q1 (identical)', 'Q2 (slightly different)', 'Q3 (very different)']
x = np.arange(len(Q_list))

plt.bar(x, kl_values, color='skyblue')
plt.xticks(x, labels, rotation=15)
plt.ylabel('KL Divergence')
plt.title('KL Divergence for different Q distributions relative to P')
plt.show()
```


##### Interpretation

- When $P$ and $Q$ are identical, KL divergence is 0.
- When $Q$ differs from $P$, KL divergence increases.
- KL divergence is not symmetric, so swapping $P$ and $Q$ changes the result. Keep this in mind when interpreting results.
- You may notice the difference between the formula in this task and the formula in the *measure_target_uniformity()* function. This is because we are comparing our probability distribution with a uniform distribution, and then the formula transforms into that form.

## Submission Files

We will only need this notebook from you - along with the key definition of the *unlearn* method, which will allow us to find the final weights of the model on which the evaluation will be performed on the class we select from the Fashion MNIST dataset.

## Constraints

We have proposed guidelines that enforce low model effectiveness on the target class. Remember that in general, the goal of our unlearning algorithm should be to achieve a state where the model behaves as if the data from the target class was never part of the training set.

Additionally, we prohibit modifying architecture hyperparameters, as well as any parameters of the classifying layer.

We expect your solution to rely on and use only standard libraries used in machine learning, such as torch, numpy, matplotlib/seaborn, scikit-learn.

# Your Solution

In [151]:
def unlearn(model, data, target_class):
    """
    Function that performs the "unlearn" operation on the model.
    :param model: Model to modify.
    :param data: Data on which to modify the model, containing
                 the full dataset.
    :param target_class: Index of the target class to unlearn from the model.
    return: Modified model.
    """
    model = deepcopy(model)
    num_epochs = 10
    
    for name, param in model.named_parameters():
        if "fc2" in name: 
            param.requires_grad_(False)

    original_params = {
        name: param.data.clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    forget_loader = data['fashion'][f'loader_{target_class}']
    retain_loader = data['fashion'][f'loader_~{target_class}']
    trainable_params = [p for n, p in model.named_parameters() if p.requires_grad]

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(trainable_params,lr=5e-5)
    LAMBDA_RETAIN = 6.0
    LAMBDA_L2     = 3.0
    LAMBDA_ENTROPY = 15.0

    retain_iter = iter(retain_loader)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for x, labels in forget_loader:
            x, labels = x.to(DEVICE), labels.to(DEVICE)
            try:
                x_r, y_r = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_loader)
                x_r, y_r = next(retain_iter)
            x_r, y_r = x_r.to(DEVICE), y_r.to(DEVICE)

            optimizer.zero_grad()

            logits_f = model(x)
            log_probs = F.log_softmax(logits_f, dim=1)
            probs = torch.exp(log_probs)
            loss_entropy = (probs * log_probs).sum(dim=1).mean()

            loss_f = -criterion(logits_f, labels)
            loss_f = torch.clamp(loss_f, min=-7.0)

            loss_r = criterion(model(x_r), y_r)

            loss_model = sum(torch.sum((p-original_params[n])**2) for n, p in model.named_parameters() if p.requires_grad)
            # print(f'Loss entropy: {loss_entropy.item()} Loss retain: {loss_r.item()} Loss model: {LAMBDA_L2 * loss_model.item()}')
            loss = LAMBDA_ENTROPY * loss_entropy + loss_f +  LAMBDA_RETAIN * loss_r + LAMBDA_L2 * loss_model
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item() * x.size(0)

        print(
            f"Epoch {epoch+1:03d} | "
            f"train={train_loss:.6f}"
        )

    return model


In [152]:
######################### DO NOT CHANGE THIS CELL ##########################
pretrained_model = LeNet(num_classes=10)
initial_state_dict = torch.load("./models/lenet_base_final.pt")
pretrained_model.load_state_dict(initial_state_dict)
pretrained_model = pretrained_model.to(DEVICE)

if not FINAL_EVALUATION_MODE:
    def prepare_data(data_dict, target_class):
        """
        Prepares data for model training.
        :param data_dict: Dictionary with data.
        :param target_class: Target class.
        :return: Tuple with data to forget and remaining data.
        """
        target_data = data_dict["fashion"][f"loader_{target_class}"]
        other_data = data_dict["fashion"][f"loader_~{target_class}"]
        return target_data, other_data
    
    target_class = 9
    unlearned_model = unlearn(pretrained_model, data_dict, target_class)
    target_loader, other_loader = prepare_data(data_dict, target_class)  
    score = evaluate_model(unlearned_model, target_loader, other_loader, initial_state_dict, DEVICE)
    print(f"Model score after unlearning: {score}")

Epoch 001 | train=-32318.019361
Epoch 002 | train=-140095.259888
Epoch 003 | train=-176524.790436
Epoch 004 | train=-184754.454529
Epoch 005 | train=-189077.816376
Epoch 006 | train=-192840.375244
Epoch 007 | train=-194253.534912
Epoch 008 | train=-196071.660553
Epoch 009 | train=-197859.664307
Epoch 010 | train=-199449.162292
KL divergence: 0.38
Classification accuracy on the remain set: 0.90
L2 distance: 1.02
Accuracy on the forget set: 0.01
Model score after unlearning: 83.05886036967055


In [102]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(unlearn, f)